# NSE Futures & Options — Data Ingestion Pipeline
**Author:** Keziya Kurian | **Assignment:** Senior Data Associate — Qode Advisors LLP

This notebook loads the Kaggle NSE F\&O dataset (2,533,210 rows) into a normalized DuckDB relational database.

## Pipeline Overview
1. Connect to DuckDB ()
2. Insert exchanges manually (NSE, BSE, MCX)
3. Extract unique instruments (328 rows) from CSV
4. Extract unique expiries (18,232 rows) from CSV
5. Load all 2,533,210 trade rows with proper FK IDs


In [ ]:
import duckdb
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded ✅")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")

## Step 1 — Connect to DuckDB and Load CSV

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath("."))
DB_FILE  = os.path.join(BASE_DIR, "fno_database.db")
CSV_FILE = os.path.join(BASE_DIR, "3mfanddo.csv")

conn = duckdb.connect(DB_FILE)
print(f"Connected to: {DB_FILE} ✅")

df = pd.read_csv(CSV_FILE)
df.columns = df.columns.str.strip()
print(f"CSV loaded: {len(df):,} rows × {len(df.columns)} columns ✅")
df.head(3)

## Step 2 — Insert Exchanges (Manual)

The CSV has no exchange column. All data is NSE.
We manually insert 3 exchanges to support multi-exchange schema (NSE, BSE, MCX).

In [ ]:
conn.execute("DELETE FROM exchanges")
conn.execute("""
    INSERT INTO exchanges (exchange_id, exchange_name) VALUES
    (1, 'NSE'),
    (2, 'BSE'),
    (3, 'MCX')
""")
result = conn.execute("SELECT * FROM exchanges").df()
print("Exchanges inserted ✅")
result

## Step 3 — Insert Instruments (328 unique rows)

Unique SYMBOL + INSTRUMENT_TYPE combinations.
 and  tagged as MCX (metal commodity proxies).

In [ ]:
conn.execute("DELETE FROM instruments")

MCX_SYMBOLS = {"HINDZINC", "NATIONALUM"}

unique_instruments = df[["INSTRUMENT","SYMBOL"]].drop_duplicates().reset_index(drop=True)
unique_instruments["instrument_id"] = unique_instruments.index + 1
unique_instruments["exchange_id"] = unique_instruments["SYMBOL"].apply(
    lambda s: 3 if s in MCX_SYMBOLS else 1
)

conn.executemany(
    "INSERT INTO instruments (instrument_id, symbol, instrument_type, exchange_id) VALUES (?,?,?,?)",
    unique_instruments[["instrument_id","SYMBOL","INSTRUMENT","exchange_id"]].values.tolist()
)

print(f"Instruments inserted: {len(unique_instruments):,} rows ✅")
print(f"MCX-tagged instruments: {unique_instruments[unique_instruments.exchange_id==3].shape[0]}")
unique_instruments[unique_instruments["exchange_id"]==3]

## Step 4 — Insert Expiries (18,232 unique rows)

Unique EXPIRY_DT + STRIKE_PR + OPTION_TYP combinations.
Dates are converted from DD-Mon-YYYY (Kaggle format) to YYYY-MM-DD.

In [ ]:
conn.execute("DELETE FROM expiries")

unique_expiries = df[["EXPIRY_DT","STRIKE_PR","OPTION_TYP"]].drop_duplicates().reset_index(drop=True)
unique_expiries["expiry_id"] = unique_expiries.index + 1
unique_expiries["EXPIRY_DT"] = pd.to_datetime(
    unique_expiries["EXPIRY_DT"], format="%d-%b-%Y"
).dt.strftime("%Y-%m-%d")

conn.executemany(
    "INSERT INTO expiries (expiry_id, expiry_date, strike_price, option_type) VALUES (?,?,?,?)",
    unique_expiries[["expiry_id","EXPIRY_DT","STRIKE_PR","OPTION_TYP"]].values.tolist()
)

print(f"Expiries inserted: {len(unique_expiries):,} rows ✅")
unique_expiries.head(5)

## Step 5 — Insert Trades (2,533,210 rows)

All rows from the CSV with proper  and  foreign keys.
Python dicts used as O(1) lookup maps for ID resolution.

In [ ]:
conn.execute("DELETE FROM trades")

instrument_map = dict(zip(
    zip(unique_instruments["SYMBOL"], unique_instruments["INSTRUMENT"]),
    unique_instruments["instrument_id"]
))
expiry_map = dict(zip(
    zip(unique_expiries["EXPIRY_DT"], unique_expiries["STRIKE_PR"], unique_expiries["OPTION_TYP"]),
    unique_expiries["expiry_id"]
))

df["EXPIRY_DT_PARSED"] = pd.to_datetime(
    df["EXPIRY_DT"], format="%d-%b-%Y"
).dt.strftime("%Y-%m-%d")

df["instrument_id"] = df.apply(lambda r: instrument_map.get((r["SYMBOL"],r["INSTRUMENT"])), axis=1)
df["expiry_id"]     = df.apply(lambda r: expiry_map.get((r["EXPIRY_DT_PARSED"],r["STRIKE_PR"],r["OPTION_TYP"])), axis=1)
df["trade_id"]      = range(1, len(df)+1)

trades_df = df[["trade_id","instrument_id","expiry_id",
               "OPEN","HIGH","LOW","CLOSE","SETTLE_PR",
               "CONTRACTS","VAL_INLAKH","OPEN_INT","CHG_IN_OI","TIMESTAMP"]].copy()
trades_df.columns = ["trade_id","instrument_id","expiry_id",
                    "open","high","low","close","settle_pr",
                    "contracts","val_inlakh","open_int","chg_in_oi","timestamp"]
trades_df["timestamp"] = pd.to_datetime(
    trades_df["timestamp"], format="%d-%b-%Y"
).dt.strftime("%Y-%m-%d")

conn.execute("INSERT INTO trades SELECT * FROM trades_df")

count = conn.execute("SELECT COUNT(*) FROM trades").fetchone()[0]
print(f"Trades inserted: {count:,} rows ✅")

## Step 6 — Verify Table Counts

In [ ]:
summary = []
for table in ["exchanges","instruments","expiries","trades"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    summary.append({"table": table, "row_count": n})

pd.DataFrame(summary)

## Step 7 — Quick Sanity Check Query

Verify the FK joins work correctly.

In [ ]:
conn.execute("""
    SELECT i.symbol, e.exchange_name, i.instrument_type,
           COUNT(t.trade_id) AS trade_count,
           ROUND(AVG(t.close), 2) AS avg_close
    FROM trades t
    JOIN instruments i ON t.instrument_id = i.instrument_id
    JOIN exchanges e ON i.exchange_id = e.exchange_id
    GROUP BY i.symbol, e.exchange_name, i.instrument_type
    ORDER BY trade_count DESC
    LIMIT 10
""").df()

In [ ]:
conn.close()
print("Database connection closed ✅")
print("Ingestion complete — fno_database.db is ready for queries.")